# Week 6: Unsupervised Learning & Clustering
## Notebook 09: Spectral Clustering & Normalized Graph Cuts

### 1. Conceptual Intuition & Theoretical Background
Centroid-based algorithms (like K-Means) fail completely on non-convex or concentric geometries (e.g., two concentric circles).

**Spectral Clustering** treats clustering as a graph partitioning problem:
1. **Affinity Graph ($W$):** Nodes are data points, and edges represent RBF similarity:
$$W_{ij} = \exp\left( -\frac{\|x_i - x_j\|^2}{2\sigma^2} \right)$$
2. **Graph Laplacian ($L$):**
$$D_{ii} = \sum_{j} W_{ij}, \quad L_{\text{sym}} = I - D^{-1/2} W D^{-1/2}$$
3. **Cheeger's Inequality & Normalized Cut (NCut):** Finding the optimal graph cut minimizing inter-cluster similarity is NP-hard. Relaxing the discrete indicator vector into continuous domain yields the generalized eigenvalue problem:
$$L v = \lambda D v$$
Clustering the first $k$ eigenvectors with standard K-Means maps complex non-linear manifolds into linearly separable subspaces.

---

### 2. Scratch Python Implementation: Spectral Clustering on Concentric Rings

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

def spectral_clustering_scratch(X, n_clusters=2, sigma=0.2):
    n_samples = len(X)
    
    # 1. Compute Pairwise RBF Similarity Matrix (Adjacency W)
    diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
    dist_sq = np.sum(diff ** 2, axis=-1)
    W = np.exp(-dist_sq / (2.0 * sigma ** 2))
    np.fill_diagonal(W, 0.0)
    
    # 2. Compute Degree Matrix D
    d = np.sum(W, axis=1)
    d_inv_sqrt = np.power(d, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    D_inv_sqrt = np.diag(d_inv_sqrt)
    
    # 3. Compute Normalized Symmetric Graph Laplacian
    L_sym = np.eye(n_samples) - D_inv_sqrt @ W @ D_inv_sqrt
    
    # 4. Eigenvalue Decomposition: select smallest k eigenvectors
    eigenvalues, eigenvectors = np.linalg.eigh(L_sym)
    
    # Sort ascending: skip trivial first eigenvector (lambda_0 = 0)
    U = eigenvectors[:, 1 : n_clusters + 1]
    
    # Row normalize to unit length
    U_norm = U / np.linalg.norm(U, axis=1, keepdims=True)
    
    # 5. Cluster projected coordinates with standard K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(U_norm)
    return cluster_labels

# Generate Concentric Circles
np.random.seed(42)
n_pts = 100
# Inner circle
r_inner = np.random.uniform(0.1, 0.5, n_pts)
theta_inner = np.random.uniform(0, 2*np.pi, n_pts)
inner_circle = np.column_stack([r_inner * np.cos(theta_inner), r_inner * np.sin(theta_inner)])

# Outer ring
r_outer = np.random.uniform(1.2, 1.6, n_pts)
theta_outer = np.random.uniform(0, 2*np.pi, n_pts)
outer_circle = np.column_stack([r_outer * np.cos(theta_outer), r_outer * np.sin(theta_outer)])

X_rings = np.vstack([inner_circle, outer_circle])

labels = spectral_clustering_scratch(X_rings, n_clusters=2, sigma=0.3)
print(f"Total Points in Concentric Dataset: {len(labels)}")
print(f"Cluster 0 Count: {np.sum(labels == 0)} | Cluster 1 Count: {np.sum(labels == 1)}")
print("Concentric manifolds successfully separated by spectral eigenvector projection!")

### 3. Step-by-Step Code Walkthrough
1. **Adjacency Matrix ($W$)**: Measures local affinity using an RBF kernel.
2. **Normalized Laplacian ($L_{\text{sym}}$)**: Bounds eigenvalues between $0 \le \lambda_i \le 2$.
3. **Fiedler Vector (2nd Eigenvector)**: The sign of the second eigenvector provides the exact continuous relaxation of the graph Minimum Normalized Cut.

---

### 4. Industry Applications & Trade-offs
- **Computer Vision Image Segmentation:** Partitioning superpixels into foreground/background objects based on pixel intensity graphs.
- **Trade-off:** Eigenvalue decomposition of $N \times N$ matrix requires $O(N^3)$, limiting scalability for datasets larger than $50,000$ points without Nyström approximation.

---

### 5. Top Interview Questions & Answers
**Q1: Why does K-Means fail on concentric circles while Spectral Clustering succeeds?**
> **Answer:** K-Means assumes spherical, linearly separable convex clusters. Spectral clustering embeds the data into the Laplacian graph eigenspace where connected manifold components become linearly separable orthogonal vectors.

**Q2: What is the role of the Fiedler vector in graph partitioning?**
> **Answer:** The Fiedler vector is the eigenvector corresponding to the second smallest eigenvalue of the Graph Laplacian. Its components partition the graph into two disconnected or weakly connected subgraphs.